# LLM Foundations

This notebook builds the mental model needed before using LLMs in applications. You do not need transformer math here; focus on the moving parts that help you reason about prompts, context windows, retrieval, cost, and failure modes.


## Learning Objectives

By the end of this notebook, you should be able to:

- Explain why LLMs are next-token prediction systems.
- Distinguish base, instruction-tuned, and chat models.
- Trace the path from raw prompt to generated text.
- Explain why context, decoding, and retrieval choices affect reliability.


## What Is An LLM?

An LLM is a neural network trained on large collections of text. During pretraining, the model learns statistical patterns that help it predict missing or next tokens. Many production models are then instruction-tuned so they behave more like helpful assistants.


> **Tip:** Compare model families by interface and behavior, not by brand name.

The table below names each model family, describes the behavior that matters for applications, and shows the prompt shape that usually goes with that family. It gives you a compact way to compare model categories before you choose a model for a product workflow. For references on these model families and their capabilities, see the [OpenAI Models Guide](https://platform.openai.com/docs/models) and the [Anthropic Claude Models Guide](https://docs.anthropic.com/en/about-claude/models).


| Model type | Behavior | Example prompt shape |
| --- | --- | --- |
| Base model | Continues text | `The capital of France is` |
| Instruction-tuned model | Follows task instructions | `Describe Paris in one sentence.` |
| Chat model | Uses role-formatted messages | `system + user + assistant messages` |

## Beyond The Model

LLM applications combine **model behavior** with **software design choices**. The model predicts text, while the application decides which instructions, tools, retrieved sources, decoding settings, and validation rules surround that prediction. This distinction matters because many production issues are system issues rather than model issues: missing context, weak retrieval, permissive prompts, or absent checks.

Read this notebook as the architecture layer for the rest of the repository. The later notebooks use tokens, vectorization, retrieval, and evaluation to make this architecture more concrete.

**An LLM call is a packaged request.**

An LLM application is easier to reason about when the model call is treated as a **packaged request** rather than a mysterious conversation. The request contains instructions, user input, optional examples, optional retrieved context, and generation settings. The model receives all of that as **tokens**. It then produces a probability distribution for the next token, selects one token according to the decoding strategy, appends it to the visible context, and repeats the process until it reaches a stop condition.

This means the application controls more than the model alone. It controls what evidence is visible, how much answer space is reserved, how deterministic the sampling should be, and whether unsupported answers are allowed. A strong LLM system therefore combines model behavior with software controls: retrieval, validation, fallbacks, monitoring, and clear product rules.


**Engineering lens.**

When debugging LLM behavior, separate three layers:
- **Input construction** → decides what the model can see
- **Model generation** → decides how text is sampled from probabilities
- **Output handling** → decides whether the answer is accepted, parsed, cited, retried, or rejected

Many problems that look like model failures are actually application design failures: missing context, conflicting instructions, no fallback path, or an output format that was never validated.


## Important Vocabulary

- **Token:** A unit of text used by the model (e.g. a word, subword, punctuation mark, or control symbol).
- **Embedding:** A vector representation of a token, sentence or document.
- **Context window:** The maximum number of tokens the model can inspect at once.
- **Logits:** Raw scores for possible next tokens before probabilities.
- **Decoding:** The process of choosing the next token from model scores.

## Workflow

```mermaid
graph LR
    A[Raw text] --> B[Tokens]
    B --> C[Token embeddings]
    C --> D[Transformer context]
    D --> E[Logits]
    E --> F[Decoding]
    F --> G[Generated token]
    G --> B
```


## Mental Model: Prediction, Not Truth Lookup

A useful first habit is to separate **language generation** from **fact lookup**. During generation, the model chooses likely next tokens based on its parameters and the visible context. That context can include retrieved documents, tool results, system instructions, examples, and the user question, but the core operation is still token prediction.

This matters in applications because the model may produce an answer that is fluent and well formatted, but still unsupported. A production system usually needs additional grounding and checks around the model: retrieval for source material, tools for current facts, validation for output structure, and fallback behavior when the evidence is weak.


## The Core Prediction Loop

LLM generation is iterative. The model does not write a full answer in one step. It predicts a distribution over the next token, chooses one token through a decoding strategy, appends that token to the context, and repeats.


The generation example keeps a list of preselected tokens so the loop is visible. A real model would compute probabilities from neural network weights at every step. Here, the important idea is **sequencing**: each selected token becomes part of the next context. That is why early tokens can steer later tokens, why long outputs can drift, and why stop conditions matter.

Follow the loop as context grows one selected token at a time.


In [ ]:
# Simulate generation one token at a time to make the prediction loop visible.
prompt_tokens = ["Explain", "token", "cost"]
generated_tokens = []
stop_token = "."

for next_token in ["in", "one", "sentence", "."]:
    context = prompt_tokens + generated_tokens
    print("Context    ->", " ".join(context))
    print("Selected   ->", next_token)
    generated_tokens.append(next_token)
    if next_token == stop_token:
        print("Stop condition met")
        break
    print()

print()
print("Final text ->", " ".join(prompt_tokens + generated_tokens))

## Transformer Flow In Application Terms

```mermaid
graph LR
    A["Application<br/>builds prompt"] --> B["Tokenizer creates<br/>token IDs"]
    B --> C["Embedding<br/>lookup"]
    C --> D["Transformer blocks<br/>update context"]
    D --> E["Logits over<br/>vocabulary"]
    E --> F["Decoder selects<br/>next token"]
    F --> G{"Stop condition met?"}
    G -- No --> D
    G -- Yes --> H["Application receives<br/>final text"]
```


You do not need to implement this flow by hand, but you do need to know where application design choices enter the system.

| Design choice | What it controls |
| --- | --- |
| Prompt structure | Input tokens visible to the model |
| Retrieval | Context available during generation |
| Decoding settings | How deterministic the output is |
| Stop conditions and output schemas | Whether the application can reliably parse the result |


## Transformer Architecture

Under the hood, these "Transformer blocks" leverage the classic encoder-decoder (or decoder-only) neural network architecture pioneered in *Vaswani et al., 2017* (diagram below). For a complete exploration of the core mathematical architecture of modern Transformer networks, see the foundational scientific paper: [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762).

<div align="center">
  <img src="assets/01-transformer-architecture.png" alt="Transformer Encoder-Decoder Architecture" width="680" style="max-width: 100%; height: auto;" />
  <p><em>Full encoder-decoder Transformer block architecture. Source: <a href="https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons / Vaswani et al., 2017</a></em></p>
</div>


Transformers use **attention** so each token can look back at earlier tokens in the sequence to resolve meaning. In the sentence `The analyst uploaded the notebook because it contains the exercise`, attention connects `it` to `the notebook`.

| Token | Resolves to / role |
| --- | --- |
| `"it"` | `"the notebook"` — pronoun resolved by earlier context |
| `"because"` | Connects the action to its reason |
| `"exercise"` | Anchors what the notebook contains |


As visualized in the scaled dot-product attention diagram below, this self-attention mechanism computes routing weights dynamically, mapping queries, keys, and values to model the relationships between different tokens in a sequence.

<div align="center">
  <img src="assets/01-self-attention.png" alt="Self-Attention Routing" width="220" style="max-width: 100%; height: auto;" />
  <p><em>Self-Attention routing mechanism. Source: <a href="https://upload.wikimedia.org/wikipedia/commons/e/e4/Self-Attention_%28Scaled_dot-product_Attention%29.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons / Vaswani et al., 2017</a></em></p>
</div>


## Decoding Settings

Greedy decoding, temperature, top-k, and top-p control how deterministic or varied generated text can be. Lower randomness is useful for structured tasks. More randomness can help brainstorming but increases risk.


### Determinism Versus Diversity

Decoding settings are **operational controls**. Greedy decoding chooses the most likely token and is useful when reproducibility matters. Temperature changes how sharply or broadly probabilities are sampled. Top-k and top-p restrict the candidate set before sampling. For a detailed reference on decoding parameters, see the [OpenAI Text Generation Guide](https://platform.openai.com/docs/guides/text-generation). In product terms, these settings decide whether the system should behave like a stable extractor, a drafting assistant, or a brainstorming partner.

The setting is not good or bad by itself. It must match the workflow. A compliance classifier, JSON extractor, or routing decision should usually be deterministic. A creative naming task or exploratory draft can tolerate more variation. The same model can feel reliable or chaotic depending on these controls.


| Setting | Determinism | Best for |
| --- | --- | --- |
| Greedy | High | Structured extraction |
| Low temperature | High | Tests and reproducible demos |
| Higher temperature | Lower | Brainstorming alternatives |
| Top-p | Medium | Bounded sampling |

## LLMs Are Not Databases

LLMs can generate plausible answers, but they are not reliable databases. Use external sources for authoritative facts, current business records, and auditable information.


### Common Misunderstandings

| Misunderstanding | Better framing |
| --- | --- |
| The model answers from a database. | The model generates likely text from parameters and context. Use databases or retrieval for authoritative facts. |
| A longer prompt is always better. | More context can help, but irrelevant context can distract the model and increase cost. |
| Temperature makes the model smarter. | Temperature changes sampling behavior. It does not add knowledge or verification. |
| Retrieval guarantees correctness. | Retrieval improves grounding only if the retrieved sources are relevant, current, and trusted. |

When debugging LLM behavior, avoid asking only if the model is good. Ask which part of the system produced the failure: prompt, retrieval, model, decoding, output validation, or monitoring.


```mermaid
graph LR
    U[User question] --> R[Retrieve sources]
    R --> P[Build grounded prompt]
    P --> L[LLM response]
    L --> C[Check and cite]
    C --> A[Answer]
```


> **Checkpoint:** Check whether you can explain each step in the generation loop. If you cannot name where context, retrieval, or decoding enters the workflow, revisit the diagram.


## Check Your Understanding

- Why can an LLM be fluent and wrong?
- What goes into a context window?
- When should an application use retrieval or an API instead of relying on model memory?
- Which decoding setting would you choose for a JSON extraction task, and why?

<details>
<summary>Show Reference Answers</summary>
    
- An LLM can be fluent and wrong because fluency comes from likely token generation, not from verified evidence.  
- A context window contains visible input: system instructions, examples, conversation history, retrieved documents, tool results, the user request, and generated output tokens.  
- Use retrieval or an API when the answer depends on current, private, auditable, or authoritative information.  
- Use greedy decoding or a very low temperature for JSON extraction because deterministic output is easier to validate and reproduce.
   
</details>
